In [1]:
print("Hey bro")

Hey bro


In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\sayan\OneDrive\Desktop\Learning\Projects\ai-resume-screening-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 211.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def get_embedding(text):
    return model.encode(text)

In [9]:
def get_embeddings(text_list):
    return model.encode(text_list)

In [4]:
import numpy as np

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [5]:
def jd_resume_similarity(jd_text, resume_text):
    jd_emb = get_embedding(jd_text)
    resume_emb = get_embedding(resume_text)

    score = cosine_similarity(jd_emb, resume_emb)
    return round(score * 100, 2)

In [6]:
def skill_match_score(jd_skills, resume_text):
    # Embed all skills at once (efficient)
    skill_embeddings = get_embeddings(jd_skills)
    resume_embedding = get_embedding(resume_text)

    scores = []

    for skill_emb in skill_embeddings:
        sim = cosine_similarity(skill_emb, resume_embedding)
        scores.append(sim)

    return round((sum(scores) / len(scores)) * 100, 2)

In [7]:
def final_score(jd_text, resume_text, jd_skills):
    jd_score = jd_resume_similarity(jd_text, resume_text)
    skill_score = skill_match_score(jd_skills, resume_text)

    final = 0.6 * jd_score + 0.4 * skill_score

    return {
        "jd_similarity": jd_score,
        "skill_match": skill_score,
        "final_score": round(final, 2)
    }

In [10]:
jd_text = "Looking for Python developer with ML and Django experience"
resume_text = "Experienced Python developer with projects in machine learning and Flask"

jd_skills = ["python", "machine learning", "django"]

result = final_score(jd_text, resume_text, jd_skills)

print(result)

{'jd_similarity': np.float32(68.13), 'skill_match': np.float32(41.03), 'final_score': np.float32(57.29)}


In [11]:
from langchain_community.document_loaders import PDFPlumberLoader

def load_resume(file_path):
    loader = PDFPlumberLoader(file_path)
    documents = loader.load()   
    
    full_text = "\n".join([doc.page_content for doc in documents])
    
    return full_text

resume_text = load_resume("SAYAN_RESUME.pdf")

In [12]:
import pdfplumber


def extract_links_from_pdf(file_path):
    links = []

    with pdfplumber.open(file_path) as pdf:
        for _, page in enumerate(pdf.pages):
            if page.annots:
                for annot in page.annots:
                    uri = annot.get("uri")
                    if uri:
                        links.append(uri)

    return links


# Usage
links = extract_links_from_pdf("SAYAN_RESUME.pdf")

In [13]:
import re

def clean_text(text):
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove unwanted special characters (keep ., /, :)
    text = re.sub(r'[^\w\s\.\:/\-]', '', text)

    # Normalize spaces again
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
def process_jd_text(jd_input):
    return jd_input.strip()


jd_text = process_jd_text("""
📌 Role: AI / Machine Learning Engineer

Company: TechNova Solutions
Location: Bangalore, India
Experience: 0–1 Years

🚀 Job Overview

We are looking for a passionate AI/ML Engineer to join our growing team. The ideal candidate should have a strong foundation in machine learning, data structures, and backend development, with the ability to build and deploy intelligent systems.

🧠 Responsibilities
Develop and deploy machine learning models for real-world applications
Work on NLP-based systems such as chatbots and recommendation engines
Design and implement REST APIs for ML services
Perform data preprocessing, feature engineering, and model evaluation
Collaborate with frontend and backend teams for full-stack integration
Optimize model performance and scalability
                          
🛠️ Required Skills
Strong proficiency in Python
Knowledge of Machine Learning and Deep Learning concepts
Experience with NLP and text processing
Familiarity with scikit-learn, TensorFlow or PyTorch
Understanding of Data Structures and Algorithms
Experience with REST APIs and backend frameworks (Flask or Django)
Basic knowledge of SQL or MongoDB
                          
⭐ Preferred Skills
Machine Learning & AI: Scikit-learn, Pandas, NumPy
Databases: MySQL, MongoDB
Development Tools: VS Code, Jupyter Notebook, Git & GitHub, Docker
                          
🎓 Qualifications
Bachelor’s degree in Computer Science, AI, or related field
Strong problem-solving and analytical skills
                          
💡 Bonus
Personal projects in AI/ML
Contributions to open-source
Experience with deployment and CI/CD
""")

In [16]:
jd_skills = ["python", "machine learning", "flask", "aws", "deep learning", "gen ai"]

result = final_score(jd_text, clean_text(resume_text), jd_skills)

print(result)

{'jd_similarity': np.float32(61.47), 'skill_match': np.float32(26.08), 'final_score': np.float32(47.31)}


In [18]:
print(resume_text)

Sayan Mondal
Bengaluru, India | +91 6366608961 | sayan.sm2024@gmail.com | Portfolio | Linkedin | GitHub
Education
CMR Institute of Technology Bengaluru, KA
Bachelor of Engineering in Computer Science Engineering (AI & ML) Dec 2022 – May 2026
St. Joseph’s Pre-University College Bengaluru, KA
PCMC Aug 2020 – Jun 2022
Projects
Enterprise Knowledge Copilot | GitHub | Live Demo Feb 2026 – Feb 2026
• Developed a RAG-powered chatbot, that uses a knowledge base to respond to company related queries and
deployed it on AWS.
• Integrated an open-source vector database (Endee Vector DB) to store and manage the embeddings.
• Implemented both Single and Hybrid Indexing strategies to improve LLM’s response accuracy and retrieval
performance.
• Tech Stack: Flask, Streamlit, Langchain, Groq API, Endee Vector DB, Docker, AWS
Multi-label Toxicity Classification System | GitHub Jan 2025 – Apr 2025
• Designed and trained a multi-label toxicity classification model to detect harmful online content across mu

In [20]:
result = final_score(jd_text, clean_text(resume_text), jd_skills)

print(result)

{'jd_similarity': np.float32(59.07), 'skill_match': np.float32(26.08), 'final_score': np.float32(45.87)}
